# Lab2: Text Pre-processing and Regular Expressions

 In this lab, you will learn essential techniques for preparing and cleaning text data using various pre-processing steps. Additionally, you will gain hands-on experience with regular expressions, a powerful tool for pattern matching and text manipulation in natural language processing (NLP) tasks.

## Setup
Install the libraries and English language resources before running the lesson examples and tasks. The following sections preserve the course notebook's order.

In [ ]:
%pip install -q nltk spacy kagglehub pandas

In [ ]:
from pathlib import Path
import nltk

# Keep NLTK resources beside the notebook in a separate data folder.
nltk_data_folder = Path(".nlp_data/nltk")
nltk_data_folder.mkdir(parents=True, exist_ok=True)
nltk.data.path.insert(0, str(nltk_data_folder.resolve()))

resources = {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab/english",
    "wordnet": "corpora/wordnet.zip",
    "stopwords": "corpora/stopwords",
}
for resource, location in resources.items():
    try:
        nltk.data.find(location)
    except LookupError:
        if not nltk.download(resource, download_dir=str(nltk_data_folder), quiet=True):
            raise RuntimeError(f"Could not download NLTK resource: {resource}")

In [ ]:
import spacy
from spacy.cli import download

if not spacy.util.is_package("en_core_web_sm"):
    download("en_core_web_sm")

_____________________________________________________________________________________________

## Regular Expressions

Regular expressions (regex) extremely useful in extracting information from any text by searching for one or more matches of a specific search pattern

https://www.regexpal.com/

#### Regular Expressions use cases

- Data pre-processing
- Pattern matching
- Text feature Engineering
- Web scraping
- Data validation
- Data extraction

An example use case is extracting all hashtags from a tweet, or getting email addresses or phone numbers from large unstructured text content.

## Regular Expressions with Python

Python provides a convenient built-in module for managing regular expressions:

> import re

We can import it as we import any other Python library.

### Important symbols in Regular Expression (For your reference)

Let’s start with the basic regular expression characters and some examples:

- (^) : Matches the expression to its right, at the start of a string before it finds a line break.
- ($) : Matches the expression to its left, at the end of a string before it finds a line break.
- (.) : Matches any character except newline.
- (a) : Matches exactly one character.
- (ab) : Matches the string ab.

----------------------------------------------------------------------------
Quantifiers:

- (a|b) : Matches expression a or b. If a is matched first, b is not checked;
- (+) : Matches the expression to its left 1 or more times;
- (*) : Matches the expression to its left 0 or more times;
- (?) : Matches the expression to its left 0 or 1 times.
----------------------------------------------------------------------------
Character Classes:

- \w : Matches alphanumeric characters, that is a-z, A-Z, 0–9, and underscore(_);
- \W : Matches non-alphanumeric characters, that is except a-z, A-Z, 0–9, and _;
- \d : Matches digits, from 0–9;
- \D : Matches any non-digits;
- \s : Matches whitespace characters, which also include the \t, \n, \r, and space characters;
- \S : Matches non-whitespace characters.
- \n : Matches a newline character;
- \t : Matches a tab character;
- \b : Matches the word boundary (or empty string) at the start and end of a word;
- \B : Matches where \b does not, that is non-word boundary.

----------------------------------------------------------------------------
Sets:

- [abc] : Matches either a, b, or c. It does not match abc;
- [a-z] : Matches any alphabet from a to z;
- [A-Z] : Matches any alphabets in capital from A to Z;
- [a\-p] : Matches a, -, or p. It matches - because \ escapes it;
- [-z] : Matches - or z;
- [a-z0–9] : Matches characters from a to z or from 0 to 9.
- [(+*)] : Special characters become literal inside a set, so this matches (, +, *, or );
- [^ab5] : Adding ^ excludes any character in the set. Here, it matches characters that are not a, b, or 5;
----------------------------------------------------------------------------


### Regular Expression functions



#### search

Search for pattern occurrences in a string using the search function of the **re module**. This function returns a match object, containing the first matched substring (or None, if it doesn’t exist) and its position inside the original string.

In [ ]:
import re

text = 'I am enjoying the NLP course.'

print(re.search("I", text)) #r"I" means it's a raw string literal
print(re.search("se.$", text)) #$ Search at the end of the string
print(re.search("am", text))
print(re.search("m", text))
print(re.search("AI", text))

#### match

The match function is similar to search, but it only tries to match the pattern at the beginning of the target string.

In [ ]:
import re
help(re.match)

In [ ]:
import re

text = 'I am enjoying the NLP course.'

print(re.search("I.*", text)) #global
print(re.match("I.*", text)) #local >> search at the start of the string

print(re.search("enjoying.*", text))
print(re.match("enjoying.*", text))

In [ ]:
import re

text = '1999 was the year I was born.'

print(re.search(r"[a-zA-Z]+", text))
print(re.match(r"[a-zA-Z]+", text))

Search does a global search, whereas match does a local search!

**Match** is often used when you need to check if the string starts with a specific pattern.


**Search** is used when you want to find a pattern anywhere within the string.

#### findall

The findall function looks for all the pattern matches in the target string (whereas search and match look only for the first occurrence).

- `re.findall()` returns a list of all matches.

- `re.finditer()` returns an iterator of match objects.

- `re.search() and re.match()` both return a single match object for the first occurrence (but differ in where they start matching: match() matches only at the start of the string, search() anywhere).

In [ ]:
import re

text1 = "a 1 b 2 c 3"
text2 = "a1 b2 c 3"
text3 = "a 1 b 222 c 3"

print(re.findall(r"\d+", text1)) #match any digit
print(re.findall(r"\d+", text2))
print(re.findall(r"\d+", text3))

#### sub

The sub is a function that finds patterns in the target string and substitute them with another string.

flags=re.I --> non-case-sensitive

count = specify number of matches you want to replace

In [ ]:
import re
help(re.sub)

In [ ]:
import re

text = 'I am enjoying the Math course.'

print(re.sub(r"Math", "NLP", text, count=1, flags=re.I)) #flags=re.I >> Ignore the case (no matter if M or m)
                                                 # count = 1 >> number of word I need it to change. count= 0 by default

__________________________________________________________________________

## Text pre-processing steps


Text preprocessing involves transforming text into a clean and consistent format that can then be fed into a model for further analysis and learning. Raw text data might contain unwanted or unimportant text due to which our results might not give efficient accuracy, and might make it hard to understand and analyze.

**The various text preprocessing steps are:**

1. Tokenization.
2. Lower casing.
3. Stop words removal.
4. Stemming.
5. Lemmatization.

Before implementing the pre-processing, let us understand the concept first.

### Tokenization

Tokenization is used in NLP to split paragraphs and sentences into smaller units that can be more easily assigned meaning.

The first step of the NLP process is gathering the data (a sentence) and breaking it into understandable parts (words).

- **Sentence Tokenization**

In [ ]:
# NLTK and its tokenizer resources were installed in the setup cells above.

In [ ]:
import nltk

text = "I'm enjoying the NLP course! I am also learning new concepts."

print(nltk.sent_tokenize(text))

- **Word Tokenzitaion**

In [ ]:
import nltk

text = "I'm enjoying the NLP course!"

print(nltk.word_tokenize(text)) #I'm is tokenized to two tokens [I,m], course! is also tokenized to two tokens [course,!]

In [ ]:
text = "I'm enjoying the NLP course!"
text.split()

### Lower Casting

Converting a word to lower case (NLP -> nlp). Words like Book and book mean the same but when not converted to the lower case those two are represented as two different words in the vector space model (resulting in more dimensions).

In [ ]:
text = "I'm enjoying the NLP course!"
text = text.lower()
print(text)

### Stemming

Stemming is basically removing the suffix from a word and reduce it to its root word. For example: “Flying” is a word and its suffix is “ing”, if we remove “ing” from “Flying” then we will get base word or root word which is “Fly”. We uses these suffix to create a new word from original stem word.

https://www.nltk.org/howto/stem.html

#### What is PorterStemmer?

It is one of the most popular stemming methods proposed in 1980. It is based on the idea that the suffixes in the English language are made up of a combination of smaller and simpler suffixes. This stemmer is known for its speed and simplicity. The main applications of Porter Stemmer include data mining and Information retrieval. However, its applications are only limited to English words. Also, the group of stems is mapped on to the same stem and the output stem is not necessarily a meaningful word. The algorithms are fairly lengthy in nature and are known to be the oldest stemmer.

> Advantage: It produces the best output as compared to other stemmers and it has less error rate.

> Limitation:  Morphological variants produced are not always real words.

In [ ]:
import nltk
from nltk.stem import PorterStemmer

ps = PorterStemmer() # create a PorterStemmer object

words = ['run','runner','running','ran','runs','easily','fairly']

for word in words:
    print(word + ' --> ' + ps.stem(word))

#### What is SnowballStemmer?

It is a stemming algorithm which is also known as the Porter2 stemming algorithm as it is a better version of the Porter Stemmer since some issues of it were fixed in this stemmer.

> Advantage: It is slightly faster computation time than porter, with a reasonably large community around it.

In [ ]:
import nltk
from nltk.stem.snowball import SnowballStemmer

sn = SnowballStemmer(language='english')

words = ['run','runner','running','ran','runs','easily','fairly']

for word in words:
    print(word+' --> '+sn.stem(word))

#### Comparing between porter and snowball

In [ ]:
words = ['generous','generation','generously','generate']

for word in words:
    print(word+' --> '+ps.stem(word))
    print(word+' --> '+sn.stem(word))
    print('---------------------------------------')

**Note**: Spacy does not provide stemming

### Lemmatization

Lemmatization is the process of grouping together the different inflected forms of a word so they can be analyzed as a single item. Lemmatization is similar to stemming but it brings context to the words. So it links words with similar meanings to one word.

> One major difference with stemming is that lemmatize takes a part of speech parameter (pos) If not supplied, the default is “noun.”

| Part of Speech          | Description                | Example Words           |
| ----------------------- | -------------------------- | ----------------------- |
| **Noun (N)**            | Person, place, thing, idea | dog, city, love, Python |
| **Verb (V)**            | Action or state            | run, eat, is, was, jump |
| **Adjective (ADJ)**     | Describes a noun           | happy, blue, fast       |
| **Adverb (ADV)**        | Describes a verb/adjective | quickly, very, well     |
| **Pronoun (PRON)**      | Replaces a noun            | he, she, it, they       |
| **Preposition (ADP)**   | Shows relationship         | in, on, under, with     |
| **Conjunction (CONJ)**  | Connects words/clauses     | and, but, or            |
| **Determiner (DET)**    | Introduces a noun          | a, an, the              |
| **Interjection (INTJ)** | Expresses emotion          | wow, oh, hey            |


In [ ]:
# WordNet was downloaded in the setup cells above.

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

text = 'I am enjoying AI courses. I am taking NLP course.'
lemmatizer = WordNetLemmatizer()

for word in nltk.word_tokenize(text):
    print(f"{word}: ", lemmatizer.lemmatize(word))

In [ ]:
# Changing pos to verb
for word in nltk.word_tokenize(text):
    print(f"{word}: ", lemmatizer.lemmatize(word, pos='v'))

You may go through Spacy's lemmatizer: https://spacy.io/api/lemmatizer

### Stop Words Removal

The words which are generally filtered out before processing a natural language are called stop words. These are actually the most common words in any language (like articles, prepositions, pronouns, conjunctions, etc) and does not add much information to the text. Examples of a few stop words in English are “the”, “a”, “an”, “so”, “what”.

#### Why do we need to remove stopwords?

By removing these words, we remove the low-level information from our text in order to give more focus to the important information. In order words, we can say that the removal of such words does not show any negative consequences on the model we train for our task.

- reduces the dataset size
- reduces the training time

#### Do we always remove stopwords? **NO!**

We do not always remove the stop words. The removal of stop words is highly dependent on the task we are performing and the goal we want to achieve. For example, if we are training a model that can perform the sentiment analysis task, we might not remove the stop words.

Movie review: “The movie was not good at all.”

Text after removal of stop words: “movie good”

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
print(nlp.Defaults.stop_words)
print(len(nlp.Defaults.stop_words))

In [ ]:
print(nlp.vocab['myself'].is_stop)
print(nlp.vocab['mystery'].is_stop)

In [ ]:
print (nlp.vocab['btw'].is_stop) #print False

In [ ]:
nlp.Defaults.stop_words.add('btw')

nlp.vocab['btw'].is_stop = True # You must update the is_stop flag for the word you added

print (nlp.vocab['btw'].is_stop)

In [ ]:
nlp.Defaults.stop_words.discard('beyond')
nlp.vocab['beyond'].is_stop = False

print(nlp.vocab['beyond'].is_stop)

### Stop words in NLTK

In [ ]:
import nltk
# Stop words were downloaded in the setup cells above.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print(len(stop_words))

In [ ]:
print(stop_words)

_________________________________________________________________________________________________________

# Text pre-processing and Regular Expression Tasks:

# Task 1:
To make you understand better about the need of text pre-processing. We will do a small project. In this project, you need to find how many hashtags and the top 10 hashtags used in the dataset.

The dataset is about people tweets about apple company on Twitter. You can find the dataset here:

https://www.kaggle.com/datasets/seriousran/appletwittersentimenttexts


### Download and Open the Dataset
Use the Apple Twitter Sentiment Texts dataset linked above. `kagglehub` downloads it and `pandas` reads the CSV.

### Import Libraries

In [ ]:
from collections import Counter
from pathlib import Path
import re

import kagglehub
import pandas as pd

### Download the Dataset
If you download the dataset manually from Kaggle, extract its CSV file(s) into a folder named `apple_tweets_data` beside this notebook. Otherwise, the code downloads it using KaggleHub.

In [ ]:
dataset_id = "seriousran/appletwittersentimenttexts"
local_folder = Path("apple_tweets_data")

if local_folder.is_dir() and any(local_folder.rglob("*.csv")):
    dataset_folder = local_folder
    print("Using the locally downloaded dataset.")
else:
    dataset_folder = Path(kagglehub.dataset_download(dataset_id))

print("Dataset folder:", dataset_folder)

### Locate and Open the CSV
The code lists CSV files before loading the dataset. If the download contains more than one CSV, set `csv_filename` to the intended file's relative path instead of selecting one arbitrarily.

In [ ]:
csv_files = sorted(dataset_folder.rglob("*.csv"))
print("Available CSV files:")
for file in csv_files:
    print("-", file.relative_to(dataset_folder))

csv_filename = None  # Set a relative filename here only if there are multiple CSVs.

if csv_filename is not None:
    csv_path = dataset_folder / csv_filename
elif len(csv_files) == 1:
    csv_path = csv_files[0]
else:
    raise ValueError("Expected one CSV file. Check the list and set csv_filename.")

try:
    df = pd.read_csv(csv_path, encoding="utf-8")
except UnicodeDecodeError:
    # Older Twitter CSV exports may use Windows-1252 rather than UTF-8.
    df = pd.read_csv(csv_path, encoding="cp1252")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

### Select the Tweet Text Column
The code looks for a column named `text`, `tweet`, `tweets`, `tweet_text`, or `content`. If none is found, inspect the displayed columns and set `text_column` to the actual tweet text column.

In [ ]:
print("Columns:", df.columns.tolist())
text_column = None  # Set the exact column name here if automatic selection is ambiguous.

if text_column is None:
    text_columns = [
        column for column in df.columns
        if column.strip().lower() in {"text", "tweet", "tweets", "tweet_text", "content"}
    ]
    if len(text_columns) != 1:
        raise ValueError("Set text_column to the column containing the tweet text.")
    text_column = text_columns[0]

if text_column not in df.columns:
    raise ValueError(f"Column not found: {text_column}")

print("Selected text column:", text_column)
df[[text_column]].head()

### Inspect Missing Text
Missing text is replaced with an empty string for hashtag extraction. All rows, including duplicates, remain in the dataset because the exercise counts occurrences in the provided data.

In [ ]:
missing_text_count = df[text_column].isna().sum()
print(f"Rows with missing text: {missing_text_count:,}")

tweets = df[text_column].fillna("").astype(str)
print(f"Rows analysed: {len(tweets):,}")

### Extract and Count Hashtags

### Define the Regular Expression
The pattern `(?<![\w#])#(\w+)` matches a `#` followed by one or more word characters.

- `(?<![\w#])` prevents a match immediately after a word character or another `#`.
- `#` matches the hashtag symbol.
- `(\w+)` captures the hashtag body: letters (including Unicode letters), digits, or underscores.

Punctuation ends a hashtag. A bare `#` is ignored. This is a simple lab extraction rule, not a full implementation of Twitter's hashtag validation. Extracting before punctuation removal preserves the hashtag markers.

In [ ]:
hashtag_pattern = re.compile(r"(?<![\w#])#(\w+)")

example_tweet = "I like #Apple and #iPhone! #APPLE is popular."
example_hashtags = hashtag_pattern.findall(example_tweet)
print("Example tweet:", example_tweet)
print("Extracted hashtags:", example_hashtags)

### Extract and Normalize Hashtags
Lowercasing combines case variants such as `#Apple` and `#APPLE`. Repeated hashtags in a single tweet count as separate occurrences.

In [ ]:
hashtags_per_tweet = tweets.apply(
    lambda tweet: [tag.lower() for tag in hashtag_pattern.findall(tweet)]
)

all_hashtags = [
    hashtag
    for tweet_hashtags in hashtags_per_tweet
    for hashtag in tweet_hashtags
]
hashtag_counts = Counter(all_hashtags)

print("First 10 extracted hashtags:", [f"#{tag}" for tag in all_hashtags[:10]])

### Report the Hashtag Counts
Total occurrences, unique hashtags, and tweets containing hashtags answer different questions, so they are reported separately.

In [ ]:
total_hashtags = len(all_hashtags)
unique_hashtags = len(hashtag_counts)
tweets_with_hashtags = sum(bool(tags) for tags in hashtags_per_tweet)

print(f"Total hashtag occurrences: {total_hashtags:,}")
print(f"Unique hashtags (ignoring case): {unique_hashtags:,}")
print(f"Tweets containing at least one hashtag: {tweets_with_hashtags:,}")

### Display the Top 10 Hashtags
Sort by frequency from highest to lowest. Alphabetical order breaks ties so the output is reproducible. If fewer than 10 unique hashtags exist, the table displays all of them.

In [ ]:
ranked_hashtags = sorted(
    hashtag_counts.items(),
    key=lambda item: (-item[1], item[0]),
)

top_10_hashtags = pd.DataFrame(
    [(f"#{tag}", count) for tag, count in ranked_hashtags[:10]],
    columns=["Hashtag", "Occurrences"],
)
top_10_hashtags.index = pd.RangeIndex(1, len(top_10_hashtags) + 1, name="Rank")
top_10_hashtags

### Summarize the Results
This cell builds the summary from the actual computed results rather than assuming which hashtag is most frequent.

In [ ]:
if ranked_hashtags:
    highest_count = ranked_hashtags[0][1]
    most_frequent = [
        f"#{tag}" for tag, count in ranked_hashtags if count == highest_count
    ]
    print(f"Found {total_hashtags:,} hashtag occurrences across {tweets_with_hashtags:,} tweets.")
    print(f"There are {unique_hashtags:,} unique hashtags after lowercasing.")
    print("Most frequent hashtag(s):", ", ".join(most_frequent))
    print(f"Occurrences of each most frequent hashtag: {highest_count:,}")
else:
    print("No hashtags matched the extraction rule in the selected text column.")

# Task 2: Using re.compile()
Given the following text: "This year is 2021"

Write a Python program that:

1. Use re.compile() to create a regular expression pattern that matches one or more digits.
2. Print the type of the compiled pattern.
3. Use the compiled pattern to replace 2021 with 2022.
4. Print the updated text.
5. Briefly explain what re.compile() does and why it is useful when working with regular expressions.


In [ ]:
text = "This year is 2021"
pattern = re.compile(r"\d+")

print("Compiled pattern type:", type(pattern))
updated_text = pattern.sub("2022", text)
print("Updated text:", updated_text)

`re.compile()` creates a reusable regular expression object. Here, `\d+` matches one or more digits, and the compiled pattern's `sub()` method replaces them with `2022`. Compiling is useful when the same pattern is applied repeatedly and keeps its methods together.

# Task 3: Using re.split()
Given the following text = "a 11 b 2 3 c 4"

Write a Python program that:

1. Use re.split() with a regular expression pattern to split the text whenever one or more digits appear.
2. Print the resulting list.
3. Briefly explain: what re.split() does.


In [ ]:
text = "a 11 b 2 3 c 4"
split_text = re.split(r"\d+", text)

print("Result:", split_text)

`re.split()` separates a string wherever the pattern matches. The digit sequences act as separators and are removed. Spaces remain in the returned pieces, and the final digit produces an empty last piece.

# Task 4: Tokenization Using spaCy

Processes the following sentence: "I'm enjoying the NLP course!"

Write a Python program that:

1. Use spaCy library and load the small English model en_core_web_sm.
2. Iterate through the text and print each token on a separate line.
3. Briefly explain: What spacy.load() does.
5. Tokenize the same text using nltk library and notice the differences compared to spaCy output.

In [ ]:
import spacy
import nltk

nlp = spacy.load("en_core_web_sm")
text = "I'm enjoying the NLP course!"

print("spaCy tokens:")
spacy_tokens = [token.text for token in nlp(text)]
for token in spacy_tokens:
    print(token)

print("\nNLTK tokens:")
nltk_tokens = nltk.word_tokenize(text)
for token in nltk_tokens:
    print(token)

print("\nSame tokens:", spacy_tokens == nltk_tokens)

`spacy.load("en_core_web_sm")` loads the installed small English model and its processing pipeline. Processing the sentence produces a `Doc`, whose tokens can be inspected individually. For this sentence, both tokenizers split `I'm` into `I` and `'m` and separate `!`. The displayed comparison checks whether their outputs match; other sentences can produce different results.